In [1]:
import pandas as pd
from custom_functions_ZMRBA import *
from pathlib import Path

#Defining input folder
input_dir = Path("ZMRBA_inputs")
output_dir = Path("ZMRBA_outputs")


In [2]:
#### LOAD INPUTS AND PARAMETERS
# Stoichiometry
RBAstoic = output_dir / 'RBA_Stoichiometry.xlsx'
df_stoich = pd.read_excel(RBAstoic)
df_stoich.index = df_stoich.id.to_list()

# Load protein sequence lengths
proteins = input_dir / 'Protein_ZM.xlsx'
df_pro = pd.read_excel(proteins,sheet_name='RBA_proteins')
df_pro.index = df_pro.Gene.to_list()

# Dummy protein's length (assigned to be the median protein length)
#from 'PROTEIN_dummy_ZM.xlsx'
NAA_dummy = 319.5

In [3]:
### Enzyme synthesis requirement coupled to metabolic reaction rate
idx = df_stoich[df_stoich.coupling_type == 'rxn_enz'].index
eqn_list = []; kapp_list = []; eqn_list_equality = [];

for i in idx:
    lhs = "v('ENZLOAD-" + df_stoich.id[i][4:] + "') * " + "kapp('" + i + "')"
    rhs = "%mu% * v('" + i + "')"
    eqn_list.append(lhs + ' =g= ' + rhs + ';')
    eqn_list_equality.append(lhs + ' =e= ' + rhs + ';')

eqn_idx = ['EnzCap'+str(i) for i in range(0, len(eqn_list))]
eqn_list = ['EnzCap'+str(i)+'.. ' + eqn_list[i] for i in range(0, len(eqn_list))]

declare_enzCapCon = output_dir / 'RBA_enzCapacityConstraints_declares.txt'
with open(declare_enzCapCon, 'w') as f:
    f.write('\n'.join(eqn_idx))

eqn_enzCapCon = output_dir / 'RBA_enzCapacityConstraints_eqns.txt'
with open(eqn_enzCapCon, 'w') as f:
    f.write('\n'.join(eqn_list))
    
eqn_list_equality = ['EnzCap'+str(i)+'.. ' + eqn_list_equality[i] for i in range(0, len(eqn_list_equality))]
equal_enzCapCon = output_dir / 'RBA_enzCapacityConstraints_eqns_equality_version.txt'
with open(equal_enzCapCon, 'w') as f:
    f.write('\n'.join(eqn_list_equality))

In [4]:
### Write protein length file (NAA)
pro_list = []
for i in df_pro.index:
    pro_list.append("'PROSYN-" + df_pro.Gene[i] + "' " + str(len(df_pro.Sequence[i])))
pro_list.append("'PROSYN-PROTDUMMY' " + str(NAA_dummy))

pro_list = ['/'] + pro_list + ['/']

prolen = output_dir / 'RBA_proteinLength.txt'
with open(prolen, 'w') as f:
    f.write('\n'.join(pro_list))

In [5]:
### Write prosyn reaction
idx = [i for i in df_stoich.index if i[:7] == 'PROSYN-']
prosyn = ["'" + i + "'" for i in idx]
prosyn = ['/'] + prosyn + ['/']

prosynrxn = output_dir / 'RBA_rxns_prosyn.txt'
with open(prosynrxn, 'w') as f:
    f.write('\n'.join(prosyn))


In [ ]:

pro_cons = []

for i in df_pro.index:
    val = df_pro.loc[i, 'vtrans (mmol/gDW/h)'] 
        
    val = float(val)
    gene = df_pro.loc[i, 'Gene']
    
    # lower bound (80% of measured)
    lower_bound = val * 0.80
    pro_cons.append(f"v.lo('PROSYN-{gene}') = {lower_bound} * %nscale%;")
    
    # allow for the model to fix bottlenecks
    pro_cons.append(f"v.up('PROSYN-{gene}') = 1e3 * %nscale%;")
    
    # Keep the waste variable open
    pro_cons.append(f"v.up('PROWASTE-{gene}') = 1e3 * %nscale%;")

procon = output_dir / 'RBA_proteinconstraints.txt'
with open(procon, 'w') as f:
    f.write('\n'.join(pro_cons))